## Notebook Description: Data Curation and Averaging of Gas and Nutrition Data

This notebook outlines a comprehensive data curation and preprocessing workflow for two datasets: 'gas' and 'nutrition'. The primary objectives include standardizing categorical variables, filtering out erroneous or incomplete records, and aggregating replicate measurements. Key methodologies involve leveraging the pandas library for data manipulation, including the use of `df.replace()` for consistent categorization (e.g., 'Breding ' to 'Breeding'), `df.str.contains()` and `df.isna()` for identifying and marking rows for deletion based on specific keywords in remark columns or missing 'methane_intensity' values, and `df.groupby().mean()` for calculating average values and replicate counts (e.g., $\text{n_replicates_gas}$, $\text{n_replicates_nutrition}$) for each unique sample identifier ($\text{id_lab}$). The outcome is a set of cleaned and averaged datasets ready for further analysis, specifically $\text{gas_clean_av2}$ and $\text{nutrition_av2}$, which condense experimental replicates into single, representative entries.

# 1.0 Libraries

In [153]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [154]:
import numpy as np
import pandas as pd

# 2.0 Data Import

In [155]:
gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/subsets_1_2_3_4_gas_sorted_by_sql.csv')
nutrition = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/subsets_1_2_3_4_nutrition_sorted_by_sql.csv')

In [156]:
gas.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
0,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,33.24064574,0,95.43754089,34.829738,Standar,NaN,NaN,NaN,NaN,NaN
1,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,32.48546917,0,89.75200777,36.194699,Standar,NaN,NaN,NaN,NaN,NaN


In [157]:
gas.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2'],
      dtype='object')

In [158]:
nutrition.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,91.930807,12.725691,87.274309,33.010522,28.489614,45.210329
1,1,2.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,92.010000,12.629062,87.370938,33.010522,27.612657,45.235119


# 3.0 Formatting

## 3.1 Drop duplicates

In [159]:
gas.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2'],
      dtype='object')

In [160]:
key = ['id_lab', 'batch', 'run', 'replication', 'syrange']

In [161]:
duplicates = gas[
    gas.duplicated(subset=key, keep=False)
].sort_values(key)

In [162]:
print(len(duplicates))
duplicates.iloc[:-10, :20]

1943


,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,batch,run,replication,syrange,sample_weight_g,undigested_dm_g,dm_incubated,digested_feed_mg
549,1,406,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,1,1.0,0.5002,0.2082,462.036651,253.836651
550,1,406,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,1,1.0,0.5002,0.2082,462.036651,253.836651
551,1,407,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,2,2.0,0.5001,0.2150,461.944280,246.944280
552,1,407,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,2,2.0,0.5001,0.2150,461.944280,246.944280
553,1,408,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,3,3.0,0.5,0.2217,461.851910,240.151910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5696,3,176,Genetic bank,F25-1667,CIAT-9578,Fabales,Fabaceae,Dioclea,sericea,Dioclea sericea,Herbaceous_legumes,23,47,2,1,176.0,0.5,0.1570,475.379911,109.379911
5697,3,177,Genetic bank,F25-1667,CIAT-9578,Fabales,Fabaceae,Dioclea,sericea,Dioclea sericea,Herbaceous_legumes,23,47,2,2,177.0,0.5002,0.3660,475.570063,0.000000
5698,3,177,Genetic bank,F25-1667,CIAT-9578,Fabales,Fabaceae,Dioclea,sericea,Dioclea sericea,Herbaceous_legumes,23,47,2,2,177.0,0.5002,0.3660,475.570063,0.000000
5699,3,178,Genetic bank,F25-1667,CIAT-9578,Fabales,Fabaceae,Dioclea,sericea,Dioclea sericea,Herbaceous_legumes,23,47,2,3,179.0,0.05,NaN,47.537991,0.000000


## 3.2 Requisitioner update

In [163]:
gas.requisitioner.unique()

array(['LMF', 'Genetic bank', 'Breeding', nan, 'Set 2', 'Breding ',
       'LMF-invivo', 'Benchmark', 'Genbank', 'Isabel Molina',
       'Mauricio Sotelo', 'Jacobo Arango/Alejandro Montoya',
       'Jacobo Arango/ Alejandro Montoya'], dtype=object)

In [164]:
# Define the mapping for functional groups
requisitioner_mapping = {
    'Breding ': 'Breeding',
    'Genbank': 'Genetic_bank',
    'Genetic bank': 'Genetic_bank'
}

# Apply the mapping to centers datt
gas['requisitioner'] = gas['requisitioner'].replace(requisitioner_mapping)
nutrition['requisitioner'] = nutrition['requisitioner'].replace(requisitioner_mapping)

## 3.3 Urocloa Taxonomy update


In [165]:
# Define the mapping for tax name
tax_name_mapping = {
    'Brachiaria humidicola': 'Urochloa humidicola',
    'Brachiaria interespecifico': 'Urochloa interespecific',
    'Brachiaria interespecifico ': 'Urochloa interespecific',
    'Urochloa interespecifico': 'Urochloa interespecific'

}

# Apply the mapping to centers datt
gas['tax_name'] = gas['tax_name'].replace(tax_name_mapping)
nutrition['tax_name'] = nutrition['tax_name'].replace(tax_name_mapping)

In [166]:
# Define the mapping for genus
genus_mapping = {
    'Brachiaria': 'Urochloa'
}

# Apply the mapping to centers datt
gas['genus'] = gas['genus'].replace(genus_mapping)
nutrition['genus'] = nutrition['genus'].replace(genus_mapping)

In [167]:
# Define the mapping for species
species_mapping = {
    'interespecifico': 'interespecific',
    ' interespecifico': 'interespecific',
}

# Apply the mapping to centers datt
gas['species'] = gas['species'].replace(species_mapping)
nutrition['species'] = nutrition['species'].replace(species_mapping)

## 3.4 Repeated samples

In [168]:
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1651"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1652"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1653"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1654"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1655"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1656"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1658"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1664"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1665"))]

# 4.0 Curation

In [169]:
remarks_columns = ['information_remarks_1','information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
                   'digest_remarks_1', 'digest_remarks_2']

# Convert columns to string type and check for 'elim' or 'descar' (case-insensitive)
conditions = []

for col in remarks_columns:
    condition = gas[col].astype(str).str.contains(
        r'elim|descar',
        case=False,
        na=False,
        regex=True
    )
    conditions.append(condition)

# Combine the remarks conditions using OR logic (any() along axis=1)
# Create a boolean series where True means 'elim' or 'descar' is found in at least one remark column
has_keywords = pd.concat(conditions, axis=1).any(axis=1)

# New condition: check if 'methane_intensity' is NaN
is_methane_intensity_null = gas['methane_intensity'].isna()

# New condition: check if 'undigested_dm_g' is NaN
is_undigested_dm_g_null = gas['undigested_dm_g'].isna()

# Combine all conditions for deletion using OR logic
final_delete_condition = has_keywords | is_methane_intensity_null | is_undigested_dm_g_null
#

# Create the 'delete' column, assigning 'yes' or 'no' based on the final condition
gas['delete'] = np.where(final_delete_condition, 'yes', 'no')

In [170]:
gas_clean = gas[gas['delete'] == 'no']

In [171]:
gas.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_complete_subsets_1234_2026_06_09.csv', index=None)

In [172]:
gas_clean.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_clean_subsets_1234_2026_06_09.csv', index=None)

In [173]:
nutrition.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/nutrition_complete_1234_2026_06_09.csv', index=None)

# 5.0 Gas

## 5.1 Mean

In [174]:
gas_clean.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete'],
      dtype='object')

In [175]:
#Columns processing
category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete']
numeric_columns = [ 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm']

for df in [gas_clean]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

/tmp/ipykernel_678/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')
/tmp/ipykernel_678/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')
/tmp/ipykernel_678/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/use

In [176]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [177]:
gas_clean_av = mean_with_replicates(gas_clean, category_columns, numeric_columns)

/tmp/ipykernel_678/1161671181.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_678/1161671181.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_678/1161671181.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [178]:
gas_clean_av2 = gas_clean_av[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'net_gas_8h_ml','net_gas_24h_ml','gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas']]

In [179]:
gas_clean_av2

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas
0,Dieta-1_Exp1,Dieta-1-Exp-1,3,839,LMF-invivo,Dieta 1_Exp1,NaN,26.17,60.87,136.67,29.35,2.15,15.67,15.45,21.11,81.06,360.94,6
1,Dieta-1_Exp2,Dieta-1-Exp-2,3,1019,LMF-invivo,Dieta 1_Exp2,NaN,20.99,43.67,98.08,17.76,1.80,14.18,14.74,14.45,76.94,342.57,5
2,Dieta-2_Exp1,Dieta-2-Exp-1,3,738,LMF-invivo,Dieta 2_Exp1,NaN,30.76,62.23,139.76,28.47,2.04,12.95,13.78,19.25,70.66,314.61,6
3,Dieta-2_Exp2,Dieta-2-Exp-2,3,1022,LMF-invivo,Dieta 2_Exp2,NaN,25.22,50.07,112.45,25.08,2.23,13.60,14.19,15.96,63.75,283.91,6
4,F24-3416,CIAT-705,1,1,Genetic_bank,Indigofera suffruticosa,Herbaceous_legumes,29.35,61.94,134.59,74.34,5.52,14.26,17.35,23.36,31.84,149.32,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
658,F26-0037,CIAT-18947,4,1048,Genetic_bank,Sesbania sesban,Shrub_Trees,35.23,73.90,154.75,51.99,3.36,13.67,13.76,21.30,41.93,-0.47,7
659,F26-0038,CIAT-19165,4,1051,Genetic_bank,Sesbania keniensis,Shrub_Trees,36.44,76.11,157.76,44.03,2.79,14.66,15.21,24.00,56.24,-8.23,7
660,F26-0039,CIAT-21899,4,1054,Genetic_bank,Sesbania sesban,Shrub_Trees,36.09,68.90,143.71,36.07,2.51,14.06,14.81,21.29,61.07,0.43,7
661,F26-0040,CIAT-23414,4,1057,Genetic_bank,Codariocalyx motorius,Shrub_Trees,29.68,59.26,122.99,26.23,2.13,14.60,15.26,18.76,73.03,-0.58,5


In [180]:
gas_clean_av2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_clean_average_subsets_1234_2026_06_09.csv', index=None)

## 5.2 Standard deviation

In [181]:
# Group by subset and id_lab, keeping categorical columns
def std_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the standard deviation of the numeric columns
    std_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .std()
          .round(2)
    )

    # Count the number of replicates in each group
    std_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, stds, and replicate counts
    result = (
        cat_df
        .join(std_df)
        .reset_index()
    )

    return result

In [182]:
gas_clean_std = std_with_replicates(gas_clean, category_columns, numeric_columns)

/tmp/ipykernel_678/2013314210.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_678/2013314210.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_678/2013314210.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [183]:
gas_clean_std2 = gas_clean_std[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'net_gas_8h_ml','net_gas_24h_ml','gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas']]

In [184]:
gas_clean_std2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_clean_std_subsets_1234_2026_06_09.csv', index=None)

#6.0 Nutrition

## 6.1 Mean

In [185]:
nutrition.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm'],
      dtype='object')

In [186]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat']
numeric_columns = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']

for df in [nutrition]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [187]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [188]:
nutrition_av = mean_with_replicates(nutrition, category_columns, numeric_columns)

/tmp/ipykernel_678/1194620962.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_678/1194620962.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_678/1194620962.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [189]:
nutrition_av.columns

Index(['id_lab', 'subset', 'no', 'requisitioner', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition'],
      dtype='object')

In [190]:
nutrition_av2 = nutrition_av[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition']]

In [191]:
nutrition_av2.head()

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
0,Dieta-1_Exp1,Dieta-1-Exp-1,3,193.0,LMF-invivo,Dieta 1_Exp1,NaN,89.00,9.50,89.53,17.68,35.48,71.52,2
1,Dieta-1_Exp2,Dieta-1-Exp-2,3,235.0,LMF-invivo,Dieta 1_Exp2,NaN,89.00,9.50,89.53,0.00,0.00,0.00,2
2,Dieta-2_Exp1,Dieta-2-Exp-1,3,195.0,LMF-invivo,Dieta 2_Exp1,NaN,89.00,9.10,89.58,13.79,33.91,67.86,2
3,Dieta-2_Exp2,Dieta-2-Exp-2,3,237.0,LMF-invivo,Dieta 2_Exp2,NaN,89.00,9.10,89.58,0.00,0.00,0.00,2
4,F24-3416,CIAT-705,1,1.0,Genetic_bank,Indigofera suffruticosa,Herbaceous_legumes,91.97,12.68,87.32,33.01,28.05,45.22,2


In [192]:
nutrition_av2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/nutrition_average_1234_2026_06_09.csv', index=None)

## 6.2 Standard deviation

In [193]:
# Group by subset and id_lab, keeping categorical columns
def std_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the standard deviation of the numeric columns
    std_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .std()
          .round(2)
    )

    # Count the number of replicates in each group
    std_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, stds, and replicate counts
    result = (
        cat_df
        .join(std_df)
        .reset_index()
    )

    return result

In [194]:
nutrition_std = std_with_replicates(nutrition, category_columns, numeric_columns)

/tmp/ipykernel_678/1995383719.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_678/1995383719.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_678/1995383719.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [195]:
nutrition_std2 = nutrition_std[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition']]

In [196]:
nutrition_std2

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
0,Dieta-1_Exp1,Dieta-1-Exp-1,3,193.0,LMF-invivo,Dieta 1_Exp1,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
1,Dieta-1_Exp2,Dieta-1-Exp-2,3,235.0,LMF-invivo,Dieta 1_Exp2,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
2,Dieta-2_Exp1,Dieta-2-Exp-1,3,195.0,LMF-invivo,Dieta 2_Exp1,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
3,Dieta-2_Exp2,Dieta-2-Exp-2,3,237.0,LMF-invivo,Dieta 2_Exp2,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
4,F24-3416,CIAT-705,1,1.0,Genetic_bank,Indigofera suffruticosa,Herbaceous_legumes,0.06,0.07,0.07,0.00,0.62,0.02,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663,F26-0038,CIAT-19165,4,NaN,Genetic_bank,Sesbania keniensis,Shrub_Trees,0.06,2.68,2.68,0.00,0.38,0.07,2
664,F26-0039,CIAT-21899,4,NaN,Genetic_bank,Sesbania sesban,Shrub_Trees,0.06,0.04,0.04,0.00,0.55,0.20,2
665,F26-0040,CIAT-23414,4,NaN,Genetic_bank,Codariocalyx motorius,Shrub_Trees,0.23,0.14,0.14,0.46,1.55,2.07,2
666,F26-0041,CIAT-23767,4,NaN,Genetic_bank,Desmodium nicaraguense,Shrub_Trees,0.06,0.07,0.07,0.00,0.90,1.68,2


In [197]:
nutrition_std2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/nutrition_std_1234_2026_06_09.csv', index=None)